# NeuroGraph-X · Week 4 · Baseline cross-validation for AAL, Harvard-Oxford, Dosenbach 160
Owner: Laiba Afridi. Protocol: memo 0006 (baseline model, unchanged), memo 0011 (no frozen measurement before the tuning freeze).

**This notebook never measures the frozen set. It has no `--stage final` cell. Do not add one.**

It produces, for each of the three atlases: one cross-validation run in `runs/run_log.jsonl` and its out-of-fold probabilities in `runs/oof/`. The timings answer memo 0011's open check (do 48 tuning runs fit in the GPU budget?).

**Input:** the `neurographx-code` dataset (repository zip with a COMMIT file), new version made from the branch `feature/laiba-w4-cpu-inference-oof`.
**Settings:** Accelerator **GPU T4**, Internet **On**. Run the cells in order.

## 1. Copy the code into a writable folder

In [ ]:
import glob, os, shutil
code_root = os.path.dirname(glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)[0])
REPO = "/kaggle/working/ngx"
if not os.path.exists(REPO):
    shutil.copytree(code_root, REPO)
os.chdir(REPO)
print("repo:", REPO, "| commit:", open("COMMIT").read().strip() if os.path.exists("COMMIT") else "unknown")

In [ ]:
!pip install -q torch_geometric==2.8.0.post1 nilearn==0.14.1 tabulate
import torch, torch_geometric; print("torch", torch.__version__, "| PyG", torch_geometric.__version__, "| GPU", torch.cuda.is_available())

## 2. Rebuild the connectomes (Aleesha's scripts, memo 0007). Step 04 downloads about 1 GB; if it stops, run the cell again and it continues.

In [ ]:
!PYTHONPATH=src python scripts/01_fetch_phenotypic.py

In [ ]:
!PYTHONPATH=src python scripts/04_fetch_all_atlases.py --quiet

In [ ]:
!PYTHONPATH=src python scripts/07_build_connectomes.py
!PYTHONPATH=src python scripts/08_verify_connectomes.py | tail -n 15

In [ ]:
!cat docs/reports/connectome_summary.md | head -n 20

**Check before going on.** Every row must match the committed report:

| Atlas | Participants | Nodes | Flat-region participants | NaN edges | Mean z |
|---|---|---|---|---|---|
| AAL | 861 | 116 | 54 | 2014 | 0.006 |
| CC200 | 861 | 200 | 15 | 3969 | 0.008 |
| Harvard-Oxford | 861 | 111 | 198 | 1055 | 0.024 |
| Dosenbach 160 | 861 | 161 | 46 | 2440 | 0.013 |

If any number differs, stop and ask before training.

## 3. Tests (all should pass, none skipped, because Kaggle has PyTorch)

In [ ]:
!PYTHONPATH=src python -m pytest -q tests/test_leakage.py tests/test_gnn_data.py tests/test_run_logger.py tests/test_inference.py tests/test_oof.py

## 4. Baseline cross-validation, three atlases (development pool only; no frozen-set graph is built)

In [ ]:
!PYTHONPATH=src python scripts/train_gnn.py --stage cv --atlas rois_aal

In [ ]:
!PYTHONPATH=src python scripts/train_gnn.py --stage cv --atlas rois_ho

In [ ]:
!PYTHONPATH=src python scripts/train_gnn.py --stage cv --atlas rois_dosenbach160

## 5. Summary of the three runs, and the memo 0011 timing check

In [ ]:
import json
runs = [json.loads(l) for l in open("runs/run_log.jsonl") if l.strip()]
new = [r for r in runs if r.get("stage") == "cv" and r.get("dataset") == "ABIDE"
       and r.get("atlas") in ("aal", "ho", "dosenbach160") and "oof" in r["metrics"]]
for r in new:
    m = r["metrics"]
    print(f"{r['atlas']:>13} | run {r['run_id']} | CV acc {m['cv_accuracy']} | balanced {m['cv_balanced_accuracy']}"
          f" | AUC {m['cv_auc']} | majority {m['cv_majority_baseline']['mean']} | epochs {m['chosen_epochs']}"
          f" | {m['cv_seconds']} s | OOF {m['oof']['n']} -> {m['oof']['file']}")
per_run = [r["metrics"]["cv_seconds"] for r in new]
if per_run:
    est = 12 * sum(per_run) + 12 * max(per_run)  # 12 settings x 3 atlases, plus CC200 at about the largest
    print(f"\nmemo 0011: 48 tuning runs take roughly {est / 3600:.1f} GPU hours at these timings")

## 6. Evidence: copy the files to download into one folder

In [ ]:
os.makedirs("/kaggle/working/evidence/oof", exist_ok=True)
for f in ["runs/run_log.jsonl", "docs/reports/connectome_summary.md"]:
    shutil.copy(f, "/kaggle/working/evidence/")
for f in glob.glob("runs/oof/*.csv"):
    shutil.copy(f, "/kaggle/working/evidence/oof/")
print(sorted(os.listdir("/kaggle/working/evidence")), sorted(os.listdir("/kaggle/working/evidence/oof")))
!tail -n 3 runs/run_log.jsonl | cut -c1-300

Use **Save Version → Quick Save**, then download everything in `/kaggle/working/evidence` from the Output panel. There is no final cell: the frozen set is measured after the tuning freeze (memo 0011).